## Ingest Sprints Data (JSON to Delta Lake)

This notebook reads the **sprints** JSON files (containing Formula 1 sprint race results - shorter races held on Saturdays at select Grand Prix weekends) from the landing volume and loads them into a Bronze Delta table.

**What's different here?** The sprints data comes as **multiple JSON files in a folder** (not a single file). Spark handles this automatically - just point `.load()` to the folder and it reads all files inside.

**Steps:**
1. **Define schema** and **read** the JSON files
2. **Enrich** with metadata columns (ingestion timestamp + source file)
3. **Write** to the Bronze Delta table `formula1.bronze.sprints`
4. **Verify** the data was written correctly

#### Loading Configuration
We import shared variables and helper functions from the `00-common` folder.

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

#### Setting Variables
We define the source file path (pointing to the `/sprints` folder) and the target table name in variables so they're easy to reuse and change.

In [0]:
source_file = f'{loding_folder_path}/sprints'
table_name = f'{catalog_name}.{bronze_schema}.sprints' # to replace the save table in write api 

#### Verify
As a final check, we read back the sprints table to confirm all data landed correctly.

#### Schema
We define the schema listing all 14 columns:
- Race info: `season`, `round`, `raceName`, `date`, `url`
- Driver/team: `driverId`, `constructorId`
- Sprint details: `grid` (starting position), `position` (finish), `positionText`, `points`, `laps`, `number`, `status`

**Note:** `points` is a `FLOAT` (decimal) because sprint races award fewer points than main races (e.g., 8, 7, 6... for top finishers).

In [0]:
from pyspark.sql.types import *
sprints_schema = StructType([
    StructField('date', StringType(), True),
    StructField('raceName', StringType(), True),
    StructField('round', IntegerType(), True),
    StructField('season', IntegerType(), True),
    StructField('url', StringType(), True),
    StructField('constructorId', StringType(), True),
    StructField('driverId', StringType(), True),
     StructField('grid', IntegerType(), True),
    StructField('laps', IntegerType(), True),
     StructField('number', IntegerType(), True),
     StructField('points', FloatType(), True),
    StructField('position', IntegerType(), True),
    StructField('positionText', StringType(), True),
    StructField('status', StringType()),
   

])

#### Read API
We use Spark's DataFrame Reader to load all JSON files from the `sprints` folder.

**What's happening:**
- `format('json')` - tells Spark the files are JSON
- `.schema(sprints_schema)` - applies our schema with 14 columns
- `.option('multiLine', True)` - handles JSON files where a single record spans multiple lines
- `.load(source_file)` - reads ALL JSON files from the `/sprints` folder

**Note:** Since `source_file` points to a folder, Spark automatically reads every JSON file inside it and combines them into one DataFrame.

In [0]:
sprints_df = (
    spark.read
     .format('json')
    # .option('Headers',True)
     .schema(sprints_schema)
     .option('multiLine', True)
     .load(source_file)
)


In [0]:
display(sprints_df)

#### Metadata
We call `add_ingestion_metadata()` to add two tracking columns:
- **`ingestion_timestamp`** - when this data was loaded
- **`source_file`** - which file each row came from (especially useful here since there are multiple files)

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)


#### Writing Delta Table
We save the enriched DataFrame as `formula1.bronze.sprints`:
- `mode('overwrite')` - replaces the entire table each run (clean reload)
- `format('delta')` - Delta format (versioning, time travel, fast queries)
- `saveAsTable(table_name)` - registers in Unity Catalog for SQL access

In [0]:
(
    sprints_final_df
    .write
    .mode('overwrite')
    .option('overwriteSchema', True)
    .format('delta')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))


In [0]:
%sql
select season, count(*)
from formula1.bronze.sprints
group by season
order by season asc